<a href="https://colab.research.google.com/github/BBWorksB/hierarchical-AviationBERT/blob/main/hierarchical_aviation_bert/notebooks/colab_runner.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Hierarchical AviationBERT — Colab Runner

Reproduces the experiments reported in *Hierarchical Transformer Architectures for ICAO ADREP Aviation Incident Classification* (Team 30, 11-785).

**Before running:**
1. Runtime → Change runtime type → **GPU** (A100 / T4 / V100 all work).
2. The notebook clones `BBWorksB/hierarchical-AviationBERT` (the fork) — if you want to push results back, set up a GitHub PAT in the last cell.

The order is: setup → smoke test (5 min) → Aviation-BERT MLM pretraining (~1 hr) → full matrix (~4–6 hr) → figures.

## 0. Check GPU + environment

In [1]:
!nvidia-smi --query-gpu=name,memory.total,memory.free --format=csv
import torch; print('torch', torch.__version__, 'cuda?', torch.cuda.is_available())

name, memory.total [MiB], memory.free [MiB]
Tesla T4, 15360 MiB, 14913 MiB
torch 2.10.0+cu128 cuda? True


## 1. Clone the fork and install dependencies

In [2]:
%cd /content
![ -d hierarchical-AviationBERT ] || git clone https://github.com/BBWorksB/hierarchical-AviationBERT.git
%cd /content/hierarchical-AviationBERT
!git pull
# If this fork doesn't yet have our code, upload the hierarchical_aviation_bert/ folder manually.
!ls -la

/content
Cloning into 'hierarchical-AviationBERT'...
remote: Enumerating objects: 53, done.
remote: Counting objects: 100% (53/53), done.
remote: Compressing objects: 100% (52/52), done.
remote: Total 53 (delta 2), reused 15 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (53/53), 2.46 MiB | 2.32 MiB/s, done.
Resolving deltas: 100% (2/2), done.
/content/hierarchical-AviationBERT
Already up to date.
total 8024
drwxr-xr-x 5 root root    4096 Apr 20 19:52 .
drwxr-xr-x 1 root root    4096 Apr 20 19:52 ..
-rw-r--r-- 1 root root   52093 Apr 20 19:52 aviation_bert_complete.ipynb
-rw-r--r-- 1 root root 2572176 Apr 20 19:52 aviTdata.csv
drwxr-xr-x 8 root root    4096 Apr 20 19:52 .git
drwxr-xr-x 8 root root    4096 Apr 20 19:52 hierarchical_aviation_bert
-rw-r--r-- 1 root root  497503 Apr 20 19:52 labeled_aviation_reports_2000.csv
-rw-r--r-- 1 root root  476565 Apr 20 19:52 labeled_aviation_reports_2001.csv
-rw-r--r-- 1 root root  423863 Apr 20 19:52 labeled_aviation_reports_2002.csv


In [3]:
!pip -q install 'transformers>=4.40' 'datasets>=2.14' 'scikit-learn>=1.3' pandas numpy matplotlib pyyaml
import sys; sys.path.insert(0, '/content/hierarchical-AviationBERT')
print('sys.path updated')

sys.path updated


## 2. Smoke test — verify every head runs on a subsample

Runs all 4 heads (FlatHead / ConstH / HierBERT / LearnableGate) on a 500-row subset for 1 epoch. Should complete in < 5 minutes on a T4. If any head errors, debug here before launching the full matrix.

In [4]:
import pandas as pd, glob, os
CSVS = sorted(glob.glob('labeled_aviation_reports_*.csv'))
print(f'Found {len(CSVS)} yearly CSVs')
# Subsample for the smoke test — save a trimmed CSV so the data loader sees fewer rows
dfs = [pd.read_csv(p, encoding='latin-1', low_memory=False) for p in CSVS]
sample = pd.concat(dfs, ignore_index=True).sample(n=500, random_state=42)
os.makedirs('tmp_smoke', exist_ok=True)
sample.to_csv('tmp_smoke/mini.csv', index=False)
print('Wrote tmp_smoke/mini.csv with 500 rows')

Found 12 yearly CSVs
Wrote tmp_smoke/mini.csv with 500 rows


In [ ]:
from hierarchical_aviation_bert.train import TrainArgs, train_one
import logging; logging.basicConfig(level=logging.INFO, format='%(asctime)s %(levelname)s %(message)s')

for head in ['flat', 'consth', 'hierbert', 'learnablegate']:
    print('=' * 60)
    print(f'Smoke test: {head}')
    args = TrainArgs(
        backbone='prajjwal1/bert-tiny',  # 2-layer BERT for quick smoke
        head=head,
        gamma=2.0,
        max_length=128,
        batch_size=8,
        grad_accum=1,
        epochs=1,
        data_csvs=['tmp_smoke/mini.csv'],
        taxonomy='hierarchical_aviation_bert/data/icao_parent_map.json',
        out=f'runs/smoke_{head}',
        fp16=False,
    )
    try:
        m = train_one(args)
        print(f'  macro_f1={m["macro_f1"]:.4f} tcr={m["tcr"]:.4f}')
    except Exception as e:
        print(f'  FAILED: {e}')
print('Smoke test done.')

Smoke test: flat


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/285 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/17.8M [00:00<?, ?B/s]

...

  FAILED: Unsupported types (<class 'NoneType'>) passed to `_pad_across_processes`. Only nested list/tuple/dicts of objects that are valid for `is_torch_tensor` should be passed.
Smoke test: consth


Loading weights:   0%|          | 0/39 [00:00<?, ?it/s]

BertModel LOAD REPORT from: prajjwal1/bert-tiny
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.decoder.bias               | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
bert.embeddings.position_ids               | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.decoder.weight             | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Epoch,Training Loss,Validation Loss


  FAILED: too many values to unpack (expected 2)
Smoke test: hierbert


## 3. Domain-adaptive pretraining — build Aviation-BERT

Continues MLM pretraining of BERT-base-uncased on the ASN narratives for 1 epoch. Saves the checkpoint to `aviation_bert_ckpt/`. On an A100 this is ~45 minutes for the full ~20k narratives.

In [ ]:
!python -m hierarchical_aviation_bert.pretrain_mlm \
    --data_csvs labeled_aviation_reports_*.csv aviTdata.csv \
    --out /content/aviation_bert_ckpt \
    --epochs 1.0 --batch 16 --max_length 256 --fp16

## 4. Full experimental matrix

{FlatHead, ConstH, HierBERT, LearnableGate} × {BERT-base, Aviation-BERT} × {γ = 0, 1, 2, 5} = 32 runs.

Each run is ~8–12 min on an A100 with 5 epochs × batch 16. Total ≈ 4–6 GPU-hours. If you need to cut budget, narrow `--gammas` to `[0 2]` and keep `--heads flat hierbert learnablegate`.

In [ ]:
!python -m hierarchical_aviation_bert.run_experiments \
    --data_csvs labeled_aviation_reports_*.csv aviTdata.csv \
    --taxonomy hierarchical_aviation_bert/data/icao_parent_map.json \
    --backbones bert-base-uncased /content/aviation_bert_ckpt \
    --heads flat consth hierbert learnablegate \
    --gammas 0.0 1.0 2.0 5.0 \
    --epochs 5 --batch_size 16 \
    --out_root runs/

## 5. Consolidated results table (Table 1 of the paper)

In [ ]:
import pandas as pd
df = pd.read_csv('runs/results_summary.csv')
print(df.to_string(index=False))
# Pick best per (head, backbone) and print as a nice comparison
best = df.loc[df.groupby(['head', 'backbone'])['macro_f1'].idxmax()]
best = best.sort_values('macro_f1', ascending=False)
print('\nBest-γ rows:')
print(best.to_string(index=False))

## 6. Regenerate report figures

In [ ]:
import json
# Real class counts from the training split (compute once from any metrics.json)
payload = json.loads(open('runs/' + sorted(os.listdir('runs'))[0] + '/metrics.json').read())
# Build class counts from the full dataset
from hierarchical_aviation_bert.data.dataset import load_asn_frame, filter_to_taxonomy
df = load_asn_frame(sorted(glob.glob('labeled_aviation_reports_*.csv')) + ['aviTdata.csv'])
df, child_names, _, _ = filter_to_taxonomy(df, 'hierarchical_aviation_bert/data/icao_parent_map.json')
counts = df['label'].value_counts().reindex(child_names).fillna(0).astype(int)
with open('runs/class_counts.json', 'w') as f: json.dump(counts.to_dict(), f)
print(counts)

In [ ]:
!python -m hierarchical_aviation_bert.make_figures \
    --runs runs/ \
    --out  figures/ \
    --class_counts_json runs/class_counts.json
!ls -lh figures/

## 7. Download everything (results + figures)

Run this to download a zip of `runs/` and `figures/` to your local machine. You'll then drop the PNGs into the Overleaf copy.

In [ ]:
!zip -r deliverables.zip runs/ figures/ -x '*/checkpoints/*' '*/pytorch_model.bin' '*/model.safetensors'
from google.colab import files
files.download('deliverables.zip')